Url Connection & Data Cleaning

In [ ]:
import io
import os
import time
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
nasa_api = os.getenv("NASA_MAP_KEY")
source = "VIIRS_NOAA20_NRT"
day_range = 3

top_10_locations = {
  "location_1": {"name": "Amazon Rainforest", "continent": "South America", "west_lon": -80.0, "south_lat": -5.0, "east_lon": -50.0, "north_lat": 5.0},
  "location_2": {"name": "California", "continent": "North America", "west_lon": -124.5, "south_lat": 32.5, "east_lon": -114.0, "north_lat": 42.0},
  "location_3": {"name": "Siberian Taiga", "continent": "Russia", "west_lon": 60.0, "south_lat": 50.0, "east_lon": 140.0, "north_lat": 70.0},
  "location_4": {"name": "New South Wales", "continent": "Austrailia", "west_lon": 141.0, "south_lat": -37.5, "east_lon": 154.0, "north_lat": -28.0},
  "location_5": {"name": "Pantanal", "continent": "Brazil", "west_lon": -62.0, "south_lat": -22.0, "east_lon": -55.0, "north_lat": -16.0},
  "location_6": {"name": "Alberta", "continent": "Canada", "west_lon": -120.0, "south_lat": 49.0, "east_lon": -110.0, "north_lat": 60.0},
  "location_7": {"name": "Mediterranean Basin", "continent": "Southern Europe", "west_lon": -10.0, "south_lat": 30.0, "east_lon": 40.0, "north_lat": 46.0},
  "location_8": {"name": "Indonesia", "continent": "Southeast Asia", "west_lon": 95.0, "south_lat": -11.0, "east_lon": 141.0, "north_lat": 6.0},
  "location_9": {"name": "Greece", "continent": "Southern Europe", "west_lon": 19.0, "south_lat": 34.0, "east_lon": 28.0, "north_lat": 42.0},
  "location_10": {"name": "Algeria", "continent": "North Africa", "west_lon": -9.0, "south_lat": 19.0, "east_lon": 12.0, "north_lat": 37.0}
}

dataframe_list = []
master_df = None

for location_key, location_data in top_10_locations.items():
  west_lon = location_data['west_lon']
  south_lat = location_data['south_lat']
  east_lon = location_data['east_lon']
  north_lat = location_data['north_lat']
  name = location_data['name']

  data_base_url = "https://firms.modaps.eosdis.nasa.gov/api/area"
  prediction_url = f"{data_base_url}/csv/{nasa_api}/{source}/{west_lon},{south_lat},{east_lon},{north_lat}/{day_range}"

  try: 
    response = requests.get(prediction_url)
    response.raise_for_status() 
    
    if "No data available" in response.text or len(response.text.strip()) == 0:
      print(f"No active fires found in {name} for this time range.")
      continue
    
    temp_df = pd.read_csv(io.StringIO(response.text))
    temp_df["location_name"] = name

    dataframe_list.append(temp_df)
    print(f"Successfully added {len(temp_df)} fire points from {name}.")

  except requests.exceptions.HTTPError as e:
    print(f"Error fetching {name}: {e}")
  
  time.sleep(0.5)

if dataframe_list:
  master_df = pd.concat(dataframe_list, ignore_index=True)
  print("\n Collection complete!")
  print(f"Total rows collected across all locations: {len(master_df)}")
else:
  print("\n No data was collected. Check your API key or coordinates.")
master_df.head()


Successfully added 92 fire points from Amazon Rainforest.
Successfully added 95 fire points from California.
Successfully added 2462 fire points from Siberian Taiga.
Successfully added 54 fire points from New South Wales.
Successfully added 875 fire points from Pantanal.
Successfully added 93 fire points from Alberta.
Successfully added 1610 fire points from Mediterranean Basin.
Successfully added 685 fire points from Indonesia.
Successfully added 75 fire points from Greece.
Successfully added 886 fire points from Algeria.

 Collection complete!
Total rows collected across all locations: 6927


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,location_name
6922,32.55473,3.16823,348.29,0.39,0.36,2026-05-29,1257,N20,VIIRS,l,2.0NRT,320.14,16.75,D,Algeria
6923,36.68004,3.12504,344.58,0.41,0.37,2026-05-29,1257,N20,VIIRS,n,2.0NRT,310.26,3.77,D,Algeria
6924,36.74629,6.25628,339.59,0.54,0.42,2026-05-29,1257,N20,VIIRS,l,2.0NRT,314.21,15.48,D,Algeria
6925,36.74788,3.08434,336.65,0.41,0.37,2026-05-29,1257,N20,VIIRS,n,2.0NRT,308.18,1.74,D,Algeria
6926,36.86727,6.98461,348.87,0.39,0.44,2026-05-29,1257,N20,VIIRS,n,2.0NRT,311.93,8.65,D,Algeria


Data Cleaning

In [ ]:
master_df.isnull().values.any() # No null values
master_df.duplicated().any() # No duplicated values
master_df.isna().sum() # No na values

latitude         0
longitude        0
bright_ti4       0
scan             0
track            0
acq_date         0
acq_time         0
satellite        0
instrument       0
confidence       0
version          0
bright_ti5       0
frp              0
daynight         0
location_name    0
dtype: int64

Data Exploration

In [91]:
variance_profile = master_df.groupby('location_name')[['frp', 'bright_ti5', 'bright_ti4']].agg(['mean','std'])
print("\n")
print(variance_profile)
print("\n")
prediction_features = master_df[['frp', 'bright_ti5', 'bright_ti4', 'track', 'scan']]
print(prediction_features.head())



                          frp             bright_ti5             bright_ti4  \
                         mean        std        mean        std        mean   
location_name                                                                 
Alberta              3.611075   5.022995  290.264731  11.737925  317.338925   
Algeria              2.925722   3.194643  293.324029   9.821650  320.589244   
Amazon Rainforest    5.440326   3.062460  292.820870   4.660839  336.686087   
California           9.363263  13.465720  290.481684   8.911901  326.504211   
Greece               1.340667   1.029570  291.837467   7.834947  310.875467   
Indonesia            5.854277   9.153807  291.920000   6.754186  325.951036   
Mediterranean Basin  4.025944  10.408684  294.301149  10.159183  319.969478   
New South Wales      3.774074   3.921971  284.400556   5.768975  318.087222   
Pantanal             5.079909   7.591880  293.123886   3.353972  324.961657   
Siberian Taiga       8.710504  19.868157  287.7113

Prediction: Fire Radiative Power

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X = master_df['frp']
y = master_df['bright_ti5']